In [0]:
# ---------------------------------------
# Import Required Libraries
# ---------------------------------------
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
%run /Workspace/Users/pothuritarun11@gmail.com/Data-Engineering-FMCG-Project/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
# Widgets
# ---------------------------------------
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
base_path = f's3://sportbar-rahul/customers.csv'
print(base_path)

In [0]:
# BRONZE LAYER
# =======================================
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
df.printSchema()
display(df.limit(10))

In [0]:
# Write to Bronze
df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")


In [0]:
# =======================================
# SILVER LAYER
# =======================================

df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)


In [0]:
# ---------------------------------------
# 1. Drop Duplicates
# ---------------------------------------
print("Rows before:", df_bronze.count())

df_silver = df_bronze.dropDuplicates(["customer_id"])

print("Rows after:", df_silver.count())

In [0]:
# 2. Trim Spaces in customer_name
# ---------------------------------------
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
# 3. Fix City Typos (IMPORTANT PART YOU MISSED)
# ---------------------------------------
df_silver = df_silver.withColumn(
    "city",
    F.when(F.col("city").isin("Bengalore", "Bengaluruu"), "Bengaluru")
     .when(F.col("city").isin("Hyderbad", "Hyderabadd"), "Hyderabad")
     .when(F.col("city").isin("NewDelhi", "NewDelhee", "NewDheli"), "New Delhi")
     .otherwise(F.col("city"))
)

In [0]:
# 4. Handle Null Values (Optional but Recommended)
# ---------------------------------------
df_silver = df_silver.fillna({
    "city": "Unknown"
})

In [0]:
# Write to Silver
# ---------------------------------------
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

print("✅ Silver table created successfully")

In [0]:
# 1. Sanity Check (City)
# -------------------------------
df_silver.select("city").distinct().show()


In [0]:
# 2. Fix Title Case Issue
# -------------------------------
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)


In [0]:
# Check result
df_silver.select("customer_name").distinct().show()

In [0]:
# -------------------------------
# 3. Handle Missing Cities
# -------------------------------
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
# Business confirmed fixes
customer_city_fix = {
    789403: "New Delhi",
    789420: "Bengaluru",
    789521: "Hyderabad",
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

In [0]:
# Apply fix
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn("city", F.coalesce("city", "fixed_city"))
    .drop("fixed_city")
)




In [0]:
# Validate
df_silver.show(truncate=False)


In [0]:
# -------------------------------
# 4. Convert customer_id to string
# -------------------------------
df_silver = df_silver.withColumn(
    "customer_id", F.col("customer_id").cast("string")
)
df_silver.printSchema()

In [0]:
# 5. Standardize Customer Attributes
# -------------------------------
df_silver = (
    df_silver
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

display(df_silver.limit(5))


In [0]:
from pyspark.sql.functions import col

df_silver = df_silver.withColumn("customer_id", col("customer_id").cast("int"))


In [0]:
# 6. Write Silver Table
# -------------------------------
df_silver.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")



In [0]:
#7. Gold Layer
# -------------------------------
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

In [0]:
df_gold = df_silver.select(
    "customer_id",
    "customer_name",
    "city",
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
df_gold.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")


In [0]:
# 8. Merge with Parent Table
# -------------------------------
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")


In [0]:

df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)


In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()